# EXP H Threshold Experiment

Threshold sweep for: `0.5, 0.45, 0.4, 0.35, 0.3, 0.25, 0.2` using the same EXP3 preprocessing/model bundle.

In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split

In [ ]:
ROOT = Path.cwd()
BUNDLE_CANDIDATES = [
    ROOT / 'exp3_logreg_cat' / 'models' / 'best_model_bundle.joblib',
    ROOT / 'exp3_xgb_ada' / 'models' / 'best_model_bundle.joblib',
    ROOT / 'V2.2.1.1' / 'exp3_logreg_cat' / 'models' / 'best_model_bundle.joblib',
    ROOT / 'V2.2.1.1' / 'exp3_xgb_ada' / 'models' / 'best_model_bundle.joblib',
]
DATA_CANDIDATES = [
    ROOT / 'merged_clinical_leftjoin.csv',
    ROOT / 'merged_clinical_dietary_anthro_leftjoin.csv',
    ROOT / 'merged_clinical_dietary_leftjoin.csv',
    ROOT / 'V2.2.1.1' / 'merged_clinical_leftjoin.csv',
    ROOT / 'V2.2.1.1' / 'merged_clinical_dietary_anthro_leftjoin.csv',
]

bundle_path = next((p for p in BUNDLE_CANDIDATES if p.exists()), None)
data_path = next((p for p in DATA_CANDIDATES if p.exists()), None)
if bundle_path is None or data_path is None:
    raise FileNotFoundError('Could not locate bundle and dataset for threshold experiment.')

bundle = joblib.load(bundle_path)
df = pd.read_csv(data_path)
target_candidates = ['hypertension', 'htn', 'target', 'label', 'outcome']
target_col = next((c for c in df.columns if c.lower() in target_candidates), None)
if target_col is None:
    raise ValueError('Target column not found.')

input_features = list(bundle['input_feature_names'])
X = df[input_features].copy()
y = df[target_col].astype(int).copy()

print('Bundle:', bundle_path)
print('Dataset:', data_path)
print('Rows:', len(df), '| Positives:', int(y.sum()))

In [ ]:
def bundle_transform(bundle_obj: dict, X_df: pd.DataFrame) -> pd.DataFrame:
    num_full = bundle_obj['num_cols_full']
    num_red = bundle_obj['num_cols_reduced']
    cat = bundle_obj['cat_cols']

    if num_full:
        num_imp = pd.DataFrame(bundle_obj['knn_imputer'].transform(X_df[num_full]), columns=num_full, index=X_df.index)
    else:
        num_imp = pd.DataFrame(index=X_df.index)

    if cat and bundle_obj.get('cat_imputer') is not None:
        cat_imp = pd.DataFrame(bundle_obj['cat_imputer'].transform(X_df[cat]), columns=cat, index=X_df.index)
    else:
        cat_imp = pd.DataFrame(index=X_df.index)

    parts = []
    if num_red:
        parts.append(bundle_obj['scaler'].transform(num_imp[num_red]))
    if cat and bundle_obj.get('ohe') is not None:
        parts.append(bundle_obj['ohe'].transform(cat_imp[cat].astype(str)))

    Xout = np.hstack(parts) if parts else np.empty((len(X_df), 0), dtype=float)
    feat_names = bundle_obj['feat_names']
    return pd.DataFrame(Xout, columns=feat_names, index=X_df.index)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_test_model = bundle_transform(bundle, X_test)
model = bundle['model']
proba = np.asarray(model.predict_proba(X_test_model))[:, 1]

thresholds = [0.5, 0.45, 0.4, 0.35, 0.3, 0.25, 0.2]
rows = []
for th in thresholds:
    pred = (proba >= th).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    rows.append({
        'threshold': th,
        'accuracy': accuracy_score(y_test, pred),
        'precision': precision_score(y_test, pred, zero_division=0),
        'recall': recall_score(y_test, pred, zero_division=0),
        'f1': f1_score(y_test, pred, zero_division=0),
        'specificity': specificity,
        'tp': int(tp), 'fp': int(fp), 'tn': int(tn), 'fn': int(fn),
    })

threshold_df = pd.DataFrame(rows).sort_values('threshold', ascending=False).reset_index(drop=True)
threshold_df

In [ ]:
plt.figure(figsize=(10, 4.5))
for metric in ['accuracy', 'precision', 'recall', 'f1', 'specificity']:
    plt.plot(threshold_df['threshold'], threshold_df[metric], marker='o', label=metric)
plt.gca().invert_xaxis()
plt.ylim(0.0, 1.0)
plt.xlabel('Threshold')
plt.ylabel('Metric value')
plt.title('Metric Behavior Across Thresholds')
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()